In [ ]:
!pip install yfinance pandas-ta scikit-learn torch joblib

import yfinance as yf
import pandas_ta as ta
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

# 1. 모델 클래스 정의
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(TimeSeriesLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LENGTH = 30

def create_sequences(x_data, y_data, seq_length):
    xs, ys = [], []
    for i in range(len(x_data) - seq_length):
        xs.append(x_data[i:(i + seq_length)])
        ys.append(y_data[i + seq_length])
    return np.array(xs), np.array(ys)

# 2. 통합 학습 파이프라인 함수
def train_and_save_currency(ticker, prefix):
    print(f"\n[{prefix.upper()}] 데이터 수집 및 학습 시작 ({ticker})")

    # 데이터 수집 및 멀티인덱스 평탄화
    df = yf.download(ticker, start='2019-01-01', end='2026-08-20', progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    df = df.dropna()

    # 보조지표 계산
    df.ta.rsi(length=14, append=True)
    df.ta.macd(fast=12, slow=26, signal=9, append=True)
    df.ta.bbands(length=20, std=2, append=True)
    df = df.dropna()

    rsi_col = [col for col in df.columns if col.startswith('RSI_')][0]
    macd_col = [col for col in df.columns if col.startswith('MACD_')][0]
    bbl_col = [col for col in df.columns if col.startswith('BBL_')][0]
    bbu_col = [col for col in df.columns if col.startswith('BBU_')][0]

    features = ['Close', rsi_col, macd_col, bbl_col, bbu_col]
    data = df[features].values

    # 스케일링
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    scaled_X = scaler_X.fit_transform(data)
    scaled_y = scaler_y.fit_transform(data[:, 0].reshape(-1, 1))

    # 데이터 분할
    X, y = create_sequences(scaled_X, scaled_y, SEQ_LENGTH)
    train_size = int(len(X) * 0.8)

    X_train_t = torch.FloatTensor(X[:train_size]).to(device)
    y_train_t = torch.FloatTensor(y[:train_size]).to(device)
    X_test_t = torch.FloatTensor(X[train_size:]).to(device)
    y_test_t = torch.FloatTensor(y[train_size:]).to(device)

    # 모델 초기화
    model = TimeSeriesLSTM(input_dim=len(features), hidden_dim=64, num_layers=2, output_dim=1).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # 조기 종료(Early Stopping) 설정
    epochs = 200
    patience = 20
    best_val_loss = float('inf')
    early_stop_counter = 0
    best_model_state = None

    for epoch in range(epochs):
        # 훈련
        model.train()
        optimizer.zero_grad()
        train_pred = model(X_train_t)
        train_loss = criterion(train_pred, y_train_t)
        train_loss.backward()
        optimizer.step()

        # 검증 (Test Loss)
        model.eval()
        with torch.no_grad():
            val_pred = model(X_test_t)
            val_loss = criterion(val_pred, y_test_t)

        # 베스트 모델 저장 및 Patience 체크
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_model_state = model.state_dict()
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss.item():.6f} | Test Loss: {val_loss.item():.6f}")

        if early_stop_counter >= patience:
            print(f"🛑 과적합 방지! Epoch {epoch+1}에서 조기 종료됨 (Best Test Loss: {best_val_loss:.6f})")
            break

    # 파일 저장 (가장 성능이 좋았던 시점의 가중치 사용)
    torch.save(best_model_state, f'{prefix}_lstm.pt')
    joblib.dump(scaler_X, f'{prefix}_scaler_X.pkl')
    joblib.dump(scaler_y, f'{prefix}_scaler_y.pkl')
    print(f"✅ {prefix} 파일 3개 저장 완료: {prefix}_lstm.pt 등")

# 3. 다중 통화 학습 실행
currency_targets = {
    "USDKRW=X": "usd",
    "JPYKRW=X": "jpy",
    "AUDKRW=X": "aud"
}

for ticker, prefix in currency_targets.items():
    train_and_save_currency(ticker, prefix)

print("\n🎉 모든 통화의 학습 및 저장이 완료되었습니다!")

In [ ]:
from google.colab import files
import os

print("\n📦 저장된 모델 파일들을 압축하는 중입니다...")

# 생성된 .pt와 .pkl 파일들을 'models.zip' 이라는 하나의 파일로 압축
os.system('zip models.zip *_lstm.pt *_scaler_X.pkl *_scaler_y.pkl')

print("✅ 압축이 완료되었습니다. 자동으로 다운로드를 시작합니다.")

# 브라우저를 통해 PC로 자동 다운로드
files.download('models.zip')